# Week 31, Day 6: Forge: Week 31 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w031/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
print("1.", 10 * (-50 + 70 - 2), "pA")
print("2. fluctuation-driven: the mean is below threshold, so spikes need fluctuations")
print("3. about", 2 * 0.020 * 2, "of spikes (2 x rate x window)")

### Exercise: Part A: the dynamic I-V fit

Complete `fit_eif(C)`: compute F = (C dV/dt - I)/C on the allowed samples, bin it by voltage from -75 to -44 mV in 1 mV bins (keeping bins with more than 30 samples), and fit the EIF form by grid search over ΔT (0.4 to 4) and VT (-58 to -44), returning a dictionary of C, gL, EL, VT, DT and Vr = -58.

In [ ]:
import numpy as np
import math
dt = 0.05

def ou(T, mu, sd, tau, seed):
    n = int(T / dt); r = np.random.default_rng(seed); x = np.empty(n); v = 0.0
    a = math.exp(-dt / tau); xi = r.normal(size=n)
    for k in range(n):
        v = a * v + math.sqrt(1 - a * a) * xi[k]; x[k] = v
    return mu + sd * x

def neuron(I, p, noise=0.0, seed=0, record=False):
    n = len(I); r = np.random.default_rng(seed); eta = r.normal(size=n) * noise / math.sqrt(dt)
    V = np.empty(n); v = p["EL"]; last = -1e9; sp = []
    for k in range(n):
        t = k * dt
        if t - last < 2.0:
            v = p["Vr"]; V[k] = v; continue
        v += dt * (-p["gL"] * (v - p["EL"]) + p["gL"] * p["DT"] * math.exp(min((v - p["VT"]) / p["DT"], 20)) + I[k] + eta[k]) / p["C"]
        if v > -30:
            sp.append(k); last = t; V[k] = 20.0; v = p["Vr"]; continue
        V[k] = v
    return (V, np.array(sp)) if record else np.array(sp) * dt

hidden = dict(C=150.0, gL=8.0, EL=-68.0, VT=-50.0, DT=1.0, Vr=-58.0)     # do not peek
I_train = ou(5000, 120, 100, 5, 21)
V_train, spike_idx = neuron(I_train, hidden, noise=10.0, seed=3, record=True)
dVdt = np.append((V_train[1:] - V_train[:-1]) / dt, 0.0)
ok = np.ones(len(V_train), bool)
for s in spike_idx:
    ok[max(0, s - 20):s + 100] = False
ok &= V_train < -42

def var_for(C):
    keep = ok & (V_train > -69) & (V_train < -67)
    return (I_train - C * dVdt)[keep].var()

def fit_eif(C):
    ...

Cs = np.arange(80, 250, 2.0)
C_var = Cs[int(np.argmin([var_for(c) for c in Cs]))]
fit_var = fit_eif(C_var)
print({k: round(float(v), 2) for k, v in fit_var.items()})

In [ ]:
import numpy as np
def test_exclusion():
    "ok excludes 1 ms before to 5 ms after each spike and voltages above -42 mV"
    assert not ok[spike_idx[0]] and not ok[spike_idx[0] + 90] and not np.any(ok & (V_train >= -42))

def test_fit():
    "the fit returns a parameter dictionary with EL within 1.5 mV of -68 and VT within 2 mV of -50"
    assert abs(fit_var["EL"] + 68) < 1.5 and abs(fit_var["VT"] + 50) < 2

test_exclusion()
test_fit()
print("All checks passed")

### Exercise: Part B: scoring the prediction

Write `coincidence(s1, s2, T)` as on Day 5. Compute the intrinsic reliability `g_int` (mean pairwise Γ across the four repeats) and the score `g_var` of your fitted model's prediction `pred_var` against the repeats.

In [ ]:
def coincidence(s1, s2, T, delta=2.0):
    ...

I_test = ou(2000, 120, 100, 5, 22)
repeats = [neuron(I_test, hidden, noise=10.0, seed=s) for s in range(4)]
g_int = ...
pred_var = neuron(I_test, fit_var)
g_var = ...
print(f"neuron fires {np.mean([len(r_) for r_ in repeats]) / 2:.1f} Hz, model {len(pred_var) / 2:.1f} Hz")
print(f"intrinsic reliability {g_int:.2f};  model {g_var:.2f}  ({g_var / g_int:.0%} of the ceiling)")

In [ ]:
import numpy as np
def test_reliability():
    "the neuron's intrinsic reliability is above 0.7"
    assert g_int > 0.7

def test_score():
    "the model's coincidence with the trials is computed and falls short of the intrinsic reliability"
    assert abs(g_var - np.mean([coincidence(pred_var, r_, 2000) for r_ in repeats])) < 1e-12 and g_var < g_int - 0.1

test_reliability()
test_score()
print("All checks passed")

### Exercise: Part C: an independent measurement

Refit with `fit_eif(150.0)`, predict the test spikes with it (`pred_mt`), and score it (`g_mt`).

In [ ]:
fit_mt = ...
pred_mt = ...
g_mt = ...
print({k: round(float(v), 2) for k, v in fit_mt.items()})
print(f"hidden: {hidden}")
print(f"score with C from the variance method {g_var:.2f}, with C from a membrane test {g_mt:.2f}, ceiling {g_int:.2f}")

In [ ]:
import numpy as np
def test_refit():
    "the refit uses C = 150 pF and scores within 0.1 of the intrinsic reliability"
    assert fit_mt["C"] == 150.0 and g_mt > g_int - 0.1 and g_mt > g_var + 0.15

test_refit()
print("All checks passed")